In [1]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from scipy.linalg import expm

from NCC_log import build_periodic_ab, pauli_basis, pauli_decomposition, phi_term, tilde_F_term

%config InlineBackend.figure_format = 'retina'
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams.update({"font.size": 18})
plt.rcParams["text.usetex"] = True


def build_log_tilde_v(n, t_total, r, epsilon, j=1.0, h=1.0, sampling="weighted"):
    a_mat, b_mat = build_periodic_ab(n, j, h)
    t = t_total / r
    s1 = expm(-1j * b_mat * t) @ expm(-1j * a_mat * t)
    u_exact = expm(-1j * (a_mat + b_mat) * t_total)

    q0 = int(np.ceil(np.log(4 * n / epsilon)))
    s0 = max(3, int(np.ceil(np.log(4 / epsilon))))
    basis = pauli_basis(n)
    phi_terms, _ = phi_term(a_mat, b_mat, s0, base_step=min(t, 0.02))
    tilde_f_terms = tilde_F_term(phi_terms, 1, s0, s0)

    order_data = {}
    eta = {}
    for order in range(2, s0 + 1):
        terms, weighted_probs, l1_norm = pauli_decomposition(tilde_f_terms[order], basis)
        probs = weighted_probs if sampling == "weighted" else np.ones(len(terms), dtype=float) / len(terms)
        order_data[order] = {"kind": "tail", "terms": terms, "probs": probs, "l1_norm": l1_norm}
        eta[order] = l1_norm * (t**order)
        antiherm_err = np.linalg.norm(tilde_f_terms[order] + tilde_f_terms[order].conj().T, ord="fro")
        if antiherm_err <= 1e-8:
            pair_terms, pair_probs, _ = pauli_decomposition(
                tilde_f_terms[order],
                basis,
                antihermitian=True,
            )
            order_data[order] = {
                "kind": "pair",
                "terms": pair_terms,
                "probs": pair_probs if sampling == "weighted" else probs,
                "l1_norm": l1_norm,
            }

    s_orders = list(range(2, s0 + 1))
    leading_orders = [s for s in s_orders if s <= 3]
    tail_orders = [s for s in s_orders if s > 3]
    eta_pair_sum = sum(eta[s] for s in leading_orders)
    theta_pair = np.arctan(eta_pair_sum)

    tilde_v = np.zeros((2**n, 2**n), dtype=complex)
    raw_weights = {}
    if eta_pair_sum > 0:
        for s in leading_orders:
            raw_weights[s] = eta[s] / eta_pair_sum
    for s in tail_orders:
        raw_weights[s] = eta[s]

    for order in s_orders:
        data = order_data[order]
        for prob, (phase, pauli) in zip(data["probs"], data["terms"]):
            if data["kind"] == "pair":
                tilde_v += raw_weights[order] * prob * expm(1j * theta_pair * (phase * pauli))
            else:
                tilde_v += raw_weights[order] * prob * (phase * pauli)
    return tilde_v, s1, u_exact


def exact_log_total_error(n, t_total, r, epsilon, j=1.0, h=1.0, sampling="weighted"):
    tilde_v, s1, u_exact = build_log_tilde_v(n, t_total, r, epsilon, j=j, h=h, sampling=sampling)
    return np.linalg.norm(np.linalg.matrix_power(tilde_v @ s1, r) - u_exact, 2)


def find_min_segments_log(n, t_total, epsilon, j=1.0, h=1.0, sampling="weighted", r_max=512):
    low = 1
    high = 1
    err_high = exact_log_total_error(n, t_total, high, epsilon, j=j, h=h, sampling=sampling)
    while err_high > epsilon and high < r_max:
        low = high
        high *= 2
        err_high = exact_log_total_error(n, t_total, high, epsilon, j=j, h=h, sampling=sampling)
    if err_high > epsilon:
        raise RuntimeError(f"failed to reach epsilon={epsilon} by r={r_max}")
    while low + 1 < high:
        mid = (low + high) // 2
        err_mid = exact_log_total_error(n, t_total, mid, epsilon, j=j, h=h, sampling=sampling)
        if err_mid <= epsilon:
            high = mid
            err_high = err_mid
        else:
            low = mid
    return high, err_high


def fit_power_law(x, y):
    coeffs = np.polyfit(np.log(x), np.log(y), 1)
    slope = coeffs[0]
    prefactor = math.exp(coeffs[1])
    return slope, prefactor


def scaled_reference(x, y0, x0, exponent):
    return y0 * (x / x0) ** exponent


def plot_panel(ax, x, y, expected_exp, xlabel, title, invert_x=False, show_theory=True):
    slope, prefactor = fit_power_law(np.array(x, dtype=float), np.array(y, dtype=float))
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    fit = prefactor * x**slope

    ax.loglog(x, y, "o", ms=8, color="#1f77b4", label="data")
    if show_theory:
        ref = scaled_reference(x, y[0], x[0], expected_exp)
        ax.loglog(x, ref, "--", lw=2.0, color="#d62728", label=rf"theory slope ${expected_exp:.3f}$")
    ax.loglog(x, fit, ":", lw=2.2, color="#2ca02c", label=rf"fit slope ${slope:.3f}$")
    ax.xaxis.set_major_locator(mticker.FixedLocator(x))
    ax.xaxis.set_major_formatter(mticker.FixedFormatter([f"{val:g}" for val in x]))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    if invert_x:
        ax.invert_xaxis()
    ax.set_xlabel(xlabel)
    ax.set_ylabel("min gate count")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(frameon=False)
    return slope


n_values = [4, 6, 8]
t_values = [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
eps_values = [5e-2, 4e-2, 3e-2, 2.5e-2, 2e-2, 1.5e-2, 1e-2]

fixed_t_for_n = 1.0
fixed_eps_for_n = 1e-2
fixed_n_for_t = 4
fixed_eps_for_t = 1e-2
fixed_n_for_eps = 4
fixed_t_for_eps = 1.0
sampling = "weighted"


In [2]:
n_gate = []
for n in n_values:
    r_min, err = find_min_segments_log(n, fixed_t_for_n, fixed_eps_for_n, sampling=sampling)
    n_gate.append(n * r_min)
    print(f"N sweep: N={n}, r_min={r_min}, err={err:.3e}, G={n*r_min}")

fig, ax = plt.subplots(figsize=(6.4, 5.2))
slope_n = plot_panel(ax, n_values, n_gate, 5 / 3, r"$N$", r"log-NCC: $N$ scaling")
fig.tight_layout()
print(f"fitted slope (N): {slope_n:.3f}")
plt.show()


N sweep: N=4, r_min=14, err=9.969e-03, G=56
N sweep: N=6, r_min=17, err=8.323e-03, G=102


KeyboardInterrupt: 

In [ ]:
t_gate = []
for t_total in t_values:
    r_min, err = find_min_segments_log(fixed_n_for_t, t_total, fixed_eps_for_t, sampling=sampling)
    t_gate.append(fixed_n_for_t * r_min)
    print(f"T sweep: T={t_total}, r_min={r_min}, err={err:.3e}, G={fixed_n_for_t*r_min}")

fig, ax = plt.subplots(figsize=(6.4, 5.2))
slope_t = plot_panel(ax, t_values, t_gate, 4 / 3, r"$T$", r"log-NCC: $T$ scaling")
fig.tight_layout()
print(f"fitted slope (T): {slope_t:.3f}")
plt.show()


In [ ]:
eps_gate = []
log_inv_eps = np.log(1 / np.array(eps_values, dtype=float))
for eps in eps_values:
    r_min, err = find_min_segments_log(fixed_n_for_eps, fixed_t_for_eps, eps, sampling=sampling)
    eps_gate.append(fixed_n_for_eps * r_min)
    print(f"eps sweep: eps={eps}, log(1/eps)={math.log(1/eps):.3f}, r_min={r_min}, err={err:.3e}, G={fixed_n_for_eps*r_min}")

fig, ax = plt.subplots(figsize=(6.4, 5.2))
slope_eps = plot_panel(
    ax,
    log_inv_eps,
    eps_gate,
    2.0,
    r"$\log(1/\epsilon)$",
    r"log-NCC: precision validation",
)
fig.tight_layout()
print(f"fitted slope vs log(1/eps): {slope_eps:.3f}")
plt.show()
